In [2]:
!pip install finnhub-python
!pip install yfinance
!pip install transformers
!pip install tensorflow

In [3]:
import yfinance
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import transformers
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from typing import Optional
import finnhub
import tensorflow as tf


recreate this paper: https://pmc.ncbi.nlm.nih.gov/articles/PMC9955765/

In [4]:
os.environ["FINNHUB_API_KEY"] = "d3c9651r01qu125a70cgd3c9651r01qu125a70d0"

In [5]:
class dataFetcher:


    def __init__(self, ticker: str):
        self.ticker = ticker
        self.data = None
    def fetch_financial_data(self, period: str, interval: str = "1d") -> pd.DataFrame:
        """Fetch historical financial data for a given ticker."""
        try:
            stock = yfinance.Ticker(self.ticker)
            hist = stock.history(period=period, interval=interval)
            return hist
        except Exception as e:
            print(f"Error fetching data for {self.ticker}: {e}")
            return pd.DataFrame()

    def fetch_financial_data(self, start_date,end_date)->pd.DataFrame:
        """Fetch historical financial data for a given ticker."""
        try:
            stock = yfinance.Ticker(self.ticker)
            hist = stock.history(start=start_date,end=end_date)
            hist=yfinance.dowload(self.ticker, start=start_date, end=end_date)
            self.data = hist

            return hist
        except Exception as e:
            print(f"Error fetching data for {self.ticker}: {e}")
            return pd.DataFrame()
    def plot_financial_data(data: pd.DataFrame, ticker: str):
        """Plot closing prices of the financial data."""
        if data.empty:
            print("No data to plot.")
            return

        plt.figure(figsize=(10, 5))
        sns.lineplot(data=data, x=data.index, y='Close')
        plt.title(f"{ticker} Closing Prices")
        plt.xlabel("Date")
        plt.ylabel("Closing Price (USD)")
        plt.grid(True)
        plt.show()

    def fetch_company_news_finhub(self, _from_date: str, _end_date:str ) -> pd.DataFrame:
        """Fetch company news from Finnhub for the given ticker and start/end dates

        Args:
            ticker: Stock ticker (e.g., 'AAPL').
            _from_date: Start date in 'YYYY-MM-DD' format.
            _end_date: End date in 'YYYY-MM-DD' format.

        Returns:
            pandas.DataFrame with columns: ['published_date', 'headline', 'url', 'summary', 'source', 'related', 'image']

        Raises:
            ValueError: if API key is not provided.
            requests.RequestException: if the HTTP request fails.
        """

        api_key =  os.getenv("FINNHUB_API_KEY")


        symbol = self.ticker.upper()
        today = datetime.today()

        start_date=_from_date
        end_date=_end_date


        client=finnhub.Client(api_key)
        rows={"published_date": [],
                    "headline":[],
                    "url":[],
                    "summary": [],
                    "source":[],
                    "related": [],
        }

        try:
            result=client.company_news(symbol, _from=start_date, to=end_date)
            for item in result:
                published = datetime.fromtimestamp(item.get("datetime")).strftime("%Y-%m-%d %H:%M:%S") if item.get("datetime") else None
                rows["published_date"].append(published)
                rows["headline"].append(item["headline"] or "")
                rows["url"].append(item["url"] or "")
                rows["summary"].append(item["summary"] or "")
                rows["source"].append(item["source"] or "")
                rows["related"].append(item["related"] or "")
                #rows["image"].append(item["image"] or "")


        except Exception as e:
            print(f"Error fetching Finnhub news for {symbol}: {e}")
            raise

        df = pd.DataFrame(rows)
        # Sort newest first and return
        if "published_date" in df.columns:
            df = df.sort_values(by="published_date", ascending=False).reset_index(drop=True)

        return df






In [6]:
def calculate_rsi(data: pd.DataFrame, rsi_period: int =14)-> pd.Series:
    """Calculate the RSI for each day in the data"""
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)

    avg_gain = gain.rolling(window=rsi_period, min_periods=1).mean()
    avg_loss = loss.rolling(window=rsi_period, min_periods=1).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


In [ ]:
SAMPLE_TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
fetcher=dataFetcher("AAPL")
#sample_data=fetcher.fetch_financial_data(start_date="2024-10-01",end_date="2024-10-31")
#sample_data=yfinance.download("MSFT", start="2024-10-01", end="2024-10-31")
#sample_data['RSI'] = calculate_rsi(sample_data)



#article_data=fetcher.fetch_company_news_finhub("2024-10-01","2024-10-31")
#goal is to merge two datasets so stock price at each date has sentiment score for that date or whichever is closest
#article_data.head()

In [65]:
#article_data.to_csv("cached_sample_article_data.csv")

In [69]:
#article_data['published_date']=article_data['published_date'].apply(lambda x: x[:11])

In [92]:
sample_data=pd.read_csv("/content/sample_data.csv")
article_data=pd.read_csv("/content/cached_sample_article_data.csv")
sample_data=sample_data.drop([0,1], axis=0)
article_data=article_data.drop(["Unnamed: 0"],axis=1)
sample_data=sample_data.reset_index(drop=True)
def set_price_col_to_datetime_index(column_name: str, df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty or column_name not in df.columns:
        return pd.DataFrame()

    df_copy = df.copy()
    df_copy[column_name] = pd.to_datetime(df_copy[column_name], errors='coerce')
    df_copy = df_copy.dropna(subset=[column_name])
    df_copy = df_copy.set_index(column_name)
    return df_copy

sample_data = set_price_col_to_datetime_index("Price",sample_data)
article_data=set_price_col_to_datetime_index("published_date",article_data)
display(sample_data.head())

,Close,High,Low,Open,Volume
Price,,,,,
2024-10-01,225.16209411621094,228.5861456381701,222.70353511874544,228.45675818805825,63285000
2024-10-02,225.72946166992188,226.31672490502262,221.98688498556473,224.84358511910924,32880600
2024-10-03,224.6245880126953,225.75930638603796,222.28548343692833,224.09704443839323,34044200
2024-10-04,225.74935913085938,226.9437971307691,223.09172964492672,226.8442543024003,37245100
2024-10-07,220.6630401611328,224.64451046166124,220.3047072265611,223.4600206171574,39505400


In [91]:
article_data.columns

Index(['Unnamed: 0', 'published_date', 'headline', 'url', 'summary', 'source',
       'related'],
      dtype='object')

In [11]:
import transformers
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
def analyze_sentiment_headines(df)->pd.DataFrame:
    if df is None or df.empty or "headlines" not in df.columns:
        return pd.DataFrame()

    df_copy = df.copy()
    df_copy["sentiment"] = pd.NA
    df_copy["sentiment_score"] = pd.NA

    for idx, row in df_copy.iterrows():
        headline = row.get("headlines", "")
        if isinstance(headline, str) and headline.strip():
            try:
                result = sentiment_model(headline[:512])  # Truncate to first 512 chars
                if result and isinstance(result, list):
                    sentiment = result[0].get("label", "NEUTRAL")
                    score = result[0].get("score", 0.0)
                    df_copy.at[idx, "sentiment"] = sentiment
                    df_copy.at[idx, "sentiment_score"] = score
            except Exception as e:
                print(f"Error analyzing sentiment for headline at index {idx}: {e}")
                continue

    return df_copy

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


In [86]:
class SentimentAnalyzer:
    """
    Analyzes sentiment of financial news headlines and creates daily aggregated features.
    """

    def __init__(self, model_name: str = "ProsusAI/finbert"):
        """
        Initialize sentiment analyzer with a financial sentiment model.

        Args:
            model_name: Hugging Face model name. Default is FinBERT for financial sentiment.
        """
        print(f"Loading sentiment model: {model_name}...")
        self.sentiment_pipeline = pipeline(
            "sentiment-analysis",
            model=model_name,
            truncation=True,
            max_length=512
        )
        print("Model loaded successfully!")

    def analyze_headline(self, headline: str) -> dict:


        if not headline or pd.isna(headline) or headline.strip() == "":
            return {'label': 'neutral', 'score': 0.0}

        try:
            result = self.sentiment_pipeline(headline[:512])[0]
            return result
        except Exception as e:
            print(f"Error analyzing headline: {e}")
            return {'label': 'neutral', 'score': 0.0}

    def convert_sentiment_to_numeric(self, label: str, score: float) -> float:


        label_lower = label.lower()
        if label_lower == 'positive':
            return score
        elif label_lower == 'negative':
            return -score
        else:  # neutral
            return 0.0

    def analyze_articles(self, article_data: pd.DataFrame, headline_column: str = 'headline') -> pd.DataFrame:


        print(f"Analyzing sentiment for {len(article_data)} articles...")

        article_data = article_data.copy()

        # Analyze each headline
        sentiments = []
        for idx, row in article_data.iterrows():
            headline = row.get(headline_column, "")
            sentiment = self.analyze_headline(headline)
            sentiments.append(sentiment)

        # Add sentiment columns
        article_data['sentiment_label'] = [s['label'] for s in sentiments]
        article_data['sentiment_score'] = [s['score'] for s in sentiments]
        article_data['sentiment_numeric'] = [
            self.convert_sentiment_to_numeric(s['label'], s['score'])
            for s in sentiments
        ]

        print("Sentiment analysis complete!")
        return article_data

    def create_daily_sentiment_features(self, article_data: pd.DataFrame) -> pd.DataFrame:
        """
        Aggregate article sentiments to daily features.

        Args:
            article_data: DataFrame with sentiment columns and datetime index

        Returns:
            DataFrame with one row per day containing sentiment features:
                - avg_sentiment: Average sentiment for the day
                - sentiment_sum: Sum of all sentiments
                - sentiment_count: Number of articles
                - sentiment_std: Standard deviation of sentiments
                - positive_ratio: Ratio of positive articles
                - negative_ratio: Ratio of negative articles
                - sentiment_momentum: Difference from previous day's avg sentiment
        """
        print("Creating daily sentiment features...")

        # Ensure index is datetime
        if not isinstance(article_data.index, pd.DatetimeIndex):
            raise ValueError("article_data must have a DatetimeIndex")

        # Extract date only (remove time component)
        article_data = article_data.copy()
        article_data['date'] = article_data.index.date

        # Group by date and calculate aggregations
        daily_sentiment = article_data.groupby('date').agg({
            'sentiment_numeric': ['mean', 'sum', 'count', 'std'],
            'sentiment_label': lambda x: (x == 'positive').sum(),  # count positive
        }).reset_index()

        # Flatten column names
        daily_sentiment.columns = ['date', 'avg_sentiment', 'sentiment_sum',
                                   'sentiment_count', 'sentiment_std', 'positive_count']

        # Calculate ratios
        daily_sentiment['positive_ratio'] = (
            daily_sentiment['positive_count'] / daily_sentiment['sentiment_count']
        )

        # Count negative articles
        negative_counts = article_data.groupby('date').apply(
            lambda x: (x['sentiment_label'] == 'negative').sum()
        ).reset_index(name='negative_count')

        daily_sentiment = daily_sentiment.merge(negative_counts, on='date', how='left')
        daily_sentiment['negative_ratio'] = (
            daily_sentiment['negative_count'] / daily_sentiment['sentiment_count']
        )

        # Fill NaN std with 0 (when only one article per day)
        daily_sentiment['sentiment_std'] = daily_sentiment['sentiment_std'].fillna(0)

        # Calculate sentiment momentum (difference from previous day)
        daily_sentiment = daily_sentiment.sort_values('date')
        daily_sentiment['sentiment_momentum'] = (
            daily_sentiment['avg_sentiment'].diff()
        )

        # First day has no momentum, fill with 0
        daily_sentiment['sentiment_momentum'] = daily_sentiment['sentiment_momentum'].fillna(0)

        # Convert date to datetime and set as index
        daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])
        daily_sentiment = daily_sentiment.set_index('date')

        print(f"Created daily sentiment features for {len(daily_sentiment)} days")
        return daily_sentiment


In [80]:
class TechnicalIndicators:
    """
    Calculate technical indicators for financial data.
    """

    @staticmethod
    def calculate_rsi(data: pd.DataFrame, column: str = 'Close', period: int = 14) -> pd.Series:
        """
        Calculate Relative Strength Index (RSI).

        Args:
            data: DataFrame with price data
            column: Column name to calculate RSI on
            period: RSI period (default 14)

        Returns:
            Series with RSI values
        """
        print(f"Calculating RSI with period={period}...")

        # Calculate price changes
        delta = data[column].diff()

        # Separate gains and losses
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()

        # Calculate RS and RSI
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))

        return rsi

    @staticmethod
    def add_technical_indicators(data: pd.DataFrame) -> pd.DataFrame:
        """
        Add technical indicators to financial data.

        Args:
            data: DataFrame with OHLCV data

        Returns:
            DataFrame with added technical indicators
        """
        data = data.copy()

        # Add RSI
        data['RSI'] = TechnicalIndicators.calculate_rsi(data)

        # You can add more indicators here
        # data['SMA_20'] = data['Close'].rolling(window=20).mean()
        # data['EMA_12'] = data['Close'].ewm(span=12).mean()

        print("Technical indicators added!")
        return data


In [79]:
class TargetEncoder:


      @staticmethod
      def create_price_direction_target(data: pd.DataFrame,
                                      price_column: str = 'Close',
                                      threshold: float = 0.01) -> pd.DataFrame:


          print(f"Creating target variable with {threshold}% threshold...")

          data = data.copy()

          # Calculate percentage change from previous day
          data['price_change_pct'] = data[price_column].pct_change() * 100

          # Create binary target: 1 if up by more than threshold, 0 otherwise
          data['target'] = (data['price_change_pct'] > threshold).astype(int)

          # For the first row (no previous day), set to NaN
          data.loc[data.index[0], 'target'] = np.nan

          print(f"Target variable created. Distribution:")
          print(data['target'].value_counts())

          return data

In [83]:
def merge_financial_and_sentiment_data(sentiment_data: pd.DataFrame, financial_data: pd.DataFrame):

    print("Merging financial and sentiment data...")

    # Ensure both have datetime index
    if not isinstance(financial_data.index, pd.DatetimeIndex):
        raise ValueError("financial_data must have DatetimeIndex")
    if not isinstance(sentiment_data.index, pd.DatetimeIndex):
        raise ValueError("sentiment_data must have DatetimeIndex")

    # Normalize indices to date only (remove time component)
    financial_data = financial_data.copy()
    sentiment_data = sentiment_data.copy()

    financial_data.index = financial_data.index.normalize()
    sentiment_data.index = sentiment_data.index.normalize()

    # Merge on index (date)
    merged_data = financial_data.merge(
        sentiment_data,
        left_index=True,
        right_index=True,
        how='left'  # Keep all financial data dates
    )

    print(f"Merged data shape: {merged_data.shape}")
    print(f"Columns: {list(merged_data.columns)}")

    # Fill NaN sentiment values with 0 (days with no news)
    sentiment_columns = ['avg_sentiment', 'sentiment_sum', 'sentiment_count',
                        'sentiment_std', 'positive_ratio', 'negative_ratio',
                        'sentiment_momentum']

    for col in sentiment_columns:
        if col in merged_data.columns:
            merged_data[col] = merged_data[col].fillna(0)

    return merged_data


,Unnamed: 0,headline,url,summary,source,related
published_date,,,,,,
2024-10-31,0,Apple Q4: Apple Intelligence And Services To D...,https://finnhub.io/api/news?id=74b52f0aad3bc39...,We predict Apple stock will rise with a new AI...,SeekingAlpha,AAPL
2024-10-31,1,Dow Jones Rallies On Surprise Jobs Report; App...,https://finnhub.io/api/news?id=545ede2bcf46766...,Dow Jones Rallies On Surprise Jobs Report; App...,DowJones,AAPL
2024-10-31,2,"AI Stocks: Tech Giants, Cloud Titans Face 'Sho...",https://finnhub.io/api/news?id=6a86fd373ba2617...,"AI Stocks: Tech Giants, Cloud Titans Face 'Sho...",DowJones,AAPL
2024-10-31,3,Apple: The World's Most Boring Way To Make Money,https://finnhub.io/api/news?id=612e5326c57388c...,Apple's financial performance remains rock sol...,SeekingAlpha,AAPL
2024-10-31,4,"Dow Jones Futures Rise; Amazon Jumps, Jobs Rep...",https://finnhub.io/api/news?id=589541c343a909b...,"Dow Jones Futures Rise; Amazon Jumps, Jobs Rep...",DowJones,AAPL


In [85]:
def main_data_func(raw_article_data, raw_financial_data):

  analyzer=SentimentAnalyzer()
  article_data=analyzer.analyze_articles(raw_article_data)
  sentiment_data=analyzer.create_daily_sentiment_features(article_data)
  financial_data=TechnicalIndicators.add_technical_indicators(raw_financial_data)
  final_financial_data=TargetEncoder.create_price_direction_target(financial_data)
  merged_data=merge_financial_and_sentiment_data(sentiment_data, final_financial_data)
  return merged_data

In [84]:
main_data_func(article_data, sample_data)